In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures # include polynomial features
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
df=pd.read_csv('/content/data.csv')

In [ ]:
df.head(2)

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.5,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.5,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA


In [ ]:
# null values
df.isnull().sum().sum()

np.int64(0)

In [ ]:
# duplicated values
df.duplicated().sum()

np.int64(0)

In [ ]:
# outliers
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  iqr = q3 - q1
  upper = q3 + 1.5*(iqr)
  lower = q1 - 1.5*(iqr)
  outliers = df[(df[col] > upper) | (df[col] < lower)]
  print(f'Total number of outliers in {col}: {len(outliers)}')
  df = df[(df[col]>=lower) & (df[col]<=upper)]

Total number of outliers in price: 240
Total number of outliers in bedrooms: 100
Total number of outliers in bathrooms: 66
Total number of outliers in sqft_living: 57
Total number of outliers in sqft_lot: 460
Total number of outliers in floors: 0
Total number of outliers in waterfront: 6
Total number of outliers in view: 234
Total number of outliers in condition: 3
Total number of outliers in sqft_above: 68
Total number of outliers in sqft_basement: 50
Total number of outliers in yr_built: 0
Total number of outliers in yr_renovated: 0


In [ ]:
# check unique vaalues in street, city, statezip and country
print(df['street'].nunique())
print(df['city'].nunique())
print(df['statezip'].nunique())

3259
43
76


In [ ]:
# drop these three columns
df.drop(columns=['street', 'city', 'statezip','date','country'], inplace=True)

In [ ]:
# separate X and y
X = df.drop(columns=['price'])
y = df['price']

In [ ]:
# train and test split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=25)

In [ ]:
#scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# create a ridge regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

Ridge()

In [ ]:
r_pred = ridge.predict(X_test)

In [ ]:
ridge = r2_score(y_test, r_pred)

In [ ]:
mean_squared_error(y_test, r_pred)

21385716089.07621

In [ ]:
#create lasso regression
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.304e+11, tolerance: 9.240e+09
  model = cd_fast.enet_coordinate_descent(


Lasso(alpha=0.1)

In [ ]:
l_pred = lasso.predict(X_test)

In [ ]:
lasso = r2_score(y_test, l_pred)

In [ ]:
mean_squared_error(y_test, l_pred)

21385921926.052517

In [ ]:
# compare r2_score both models
df_compare = pd.DataFrame({'Model': ['Ridge', 'Lasso'], 'R2_Score': [ridge, lasso]})
df_compare

,Model,R2_Score
0,Ridge,0.424996
1,Lasso,0.424991
